# Measure RIR Model Performance

This notebook compares multiple RIR prediction methods from saved `.npz` files.

Expected file contract:
- required arrays: `y_true`, `y_pred`
- optional arrays: `sample_id`, `method_name`, `sample_rate`
- default shapes: `(n_samples, n_points)`

It reports waveform metrics (`MAE`, `RMSE`, `R^2`) and acoustic-parameter metrics for `T60`, `EDT`, `C50`, and `D50`.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

RESULT_FILES = [
    # Path("./results/method_a_predictions.npz"),
    # Path("./results/method_b_predictions.npz"),
]
SAMPLE_RATE = 16000
EPSILON = 1e-12
ENERGY_FLOOR_DB = -60.0
PLOT_EXAMPLES = True
EXAMPLE_INDEX = 0
EXPORT_CSV = False
OUTPUT_DIR = Path("./results")

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
def _as_path(path_like):
    return path_like if isinstance(path_like, Path) else Path(path_like)


def _ensure_2d(array, name, file_path):
    array = np.asarray(array)
    if array.ndim == 1:
        array = array[None, :]
    if array.ndim != 2:
        raise ValueError(
            f"{file_path}: '{name}' must be 2D after normalization, got shape {array.shape}"
        )
    return array.astype(np.float64, copy=False)


def _coerce_method_name(raw_name, file_path):
    if raw_name is None:
        return Path(file_path).stem
    if isinstance(raw_name, np.ndarray):
        if raw_name.shape == ():
            return str(raw_name.item())
        flat = raw_name.reshape(-1)
        if len(flat) == 1:
            return str(flat[0])
        return str(Path(file_path).stem)
    return str(raw_name)


def _coerce_sample_rate(raw_sample_rate, default_rate):
    if raw_sample_rate is None:
        return int(default_rate)
    if isinstance(raw_sample_rate, np.ndarray):
        if raw_sample_rate.shape == ():
            return int(raw_sample_rate.item())
        flat = raw_sample_rate.reshape(-1)
        if len(flat) == 1:
            return int(flat[0])
    return int(raw_sample_rate)


def _coerce_sample_ids(raw_sample_ids, n_samples, file_path):
    if raw_sample_ids is None:
        return np.array([f"sample_{idx:05d}" for idx in range(n_samples)], dtype=object)
    sample_ids = np.asarray(raw_sample_ids)
    sample_ids = sample_ids.reshape(-1)
    if len(sample_ids) != n_samples:
        raise ValueError(
            f"{file_path}: 'sample_id' length {len(sample_ids)} does not match n_samples {n_samples}"
        )
    return sample_ids.astype(object)


def load_result_file(file_path, default_rate=SAMPLE_RATE):
    file_path = _as_path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Result file not found: {file_path}")

    with np.load(file_path, allow_pickle=True) as data:
        keys = set(data.files)
        required = {"y_true", "y_pred"}
        missing = required - keys
        if missing:
            missing_str = ", ".join(sorted(missing))
            raise ValueError(f"{file_path}: missing required arrays: {missing_str}")

        y_true = _ensure_2d(data["y_true"], "y_true", file_path)
        y_pred = _ensure_2d(data["y_pred"], "y_pred", file_path)
        if y_true.shape != y_pred.shape:
            raise ValueError(
                f"{file_path}: y_true shape {y_true.shape} does not match y_pred shape {y_pred.shape}"
            )

        method_name = _coerce_method_name(data["method_name"] if "method_name" in keys else None, file_path)
        sample_rate = _coerce_sample_rate(data["sample_rate"] if "sample_rate" in keys else None, default_rate)
        sample_ids = _coerce_sample_ids(data["sample_id"] if "sample_id" in keys else None, y_true.shape[0], file_path)

    return {
        "file_path": file_path,
        "method_name": method_name,
        "sample_rate": sample_rate,
        "sample_id": sample_ids,
        "y_true": y_true,
        "y_pred": y_pred,
    }


def load_all_results(result_files, default_rate=SAMPLE_RATE):
    loaded = []
    for file_path in result_files:
        loaded.append(load_result_file(file_path, default_rate=default_rate))
    return loaded


In [ ]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def r2_score(y_true, y_pred, eps=EPSILON):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot <= eps:
        return np.nan
    return float(1.0 - (ss_res / ss_tot))


def compute_waveform_metrics(y_true, y_pred):
    flat_true = y_true.reshape(-1)
    flat_pred = y_pred.reshape(-1)
    per_sample_mae = np.mean(np.abs(y_true - y_pred), axis=1)
    per_sample_rmse = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=1))
    per_sample_r2 = np.array([r2_score(t, p) for t, p in zip(y_true, y_pred)], dtype=np.float64)

    return {
        "waveform_mae": mae(flat_true, flat_pred),
        "waveform_rmse": rmse(flat_true, flat_pred),
        "waveform_r2": r2_score(flat_true, flat_pred),
        "sample_mae_mean": float(np.nanmean(per_sample_mae)),
        "sample_mae_std": float(np.nanstd(per_sample_mae)),
        "sample_rmse_mean": float(np.nanmean(per_sample_rmse)),
        "sample_rmse_std": float(np.nanstd(per_sample_rmse)),
        "sample_r2_mean": float(np.nanmean(per_sample_r2)),
        "sample_r2_std": float(np.nanstd(per_sample_r2)),
        "n_samples": int(y_true.shape[0]),
        "n_points": int(y_true.shape[1]),
    }


In [ ]:
PARAMETERS = ("T60", "EDT", "C50", "D50")


def schroeder_decay_db(rir, eps=EPSILON, floor_db=ENERGY_FLOOR_DB):
    rir = np.asarray(rir, dtype=np.float64)
    energy = np.square(rir)
    if not np.any(energy > 0):
        raise ValueError("RIR energy is zero everywhere")
    decay = np.cumsum(energy[::-1])[::-1]
    decay /= max(decay[0], eps)
    decay_db = 10.0 * np.log10(np.maximum(decay, eps))
    return np.maximum(decay_db, floor_db)


def fit_decay_time(decay_db, sample_rate, upper_db, lower_db, multiplier, min_points=8):
    idx = np.where((decay_db <= upper_db) & (decay_db >= lower_db))[0]
    if len(idx) < min_points:
        raise ValueError(
            f"Not enough decay points between {upper_db} dB and {lower_db} dB: {len(idx)}"
        )
    times = idx / float(sample_rate)
    selected_db = decay_db[idx]
    slope, intercept, _, _, _ = stats.linregress(times, selected_db)
    if slope >= 0 or np.isclose(slope, 0.0):
        raise ValueError(f"Invalid decay slope: {slope}")
    return float(-multiplier / slope), float(slope), float(intercept)


def compute_clarity_definition(rir, sample_rate, split_ms=50.0, eps=EPSILON):
    rir = np.asarray(rir, dtype=np.float64)
    energy = np.square(rir)
    split_index = int(round((split_ms / 1000.0) * sample_rate))
    split_index = max(1, min(split_index, len(energy)))
    early = np.sum(energy[:split_index])
    late = np.sum(energy[split_index:])
    total = early + late
    c50 = 10.0 * np.log10((early + eps) / (late + eps))
    d50 = float(early / max(total, eps))
    return float(c50), d50


def extract_acoustic_parameters(rir, sample_rate):
    decay_db = schroeder_decay_db(rir)
    t60, _, _ = fit_decay_time(decay_db, sample_rate, upper_db=-5.0, lower_db=-35.0, multiplier=60.0)
    edt, _, _ = fit_decay_time(decay_db, sample_rate, upper_db=0.0, lower_db=-10.0, multiplier=60.0)
    c50, d50 = compute_clarity_definition(rir, sample_rate, split_ms=50.0)
    return {
        "T60": t60,
        "EDT": edt,
        "C50": c50,
        "D50": d50,
        "decay_db": decay_db,
    }


def extract_parameter_rows(method_result):
    rows = []
    invalid = []
    for idx, (sample_id, y_true, y_pred) in enumerate(
        zip(method_result["sample_id"], method_result["y_true"], method_result["y_pred"])
    ):
        try:
            true_params = extract_acoustic_parameters(y_true, method_result["sample_rate"])
            pred_params = extract_acoustic_parameters(y_pred, method_result["sample_rate"])
        except ValueError as exc:
            invalid.append({
                "method_name": method_result["method_name"],
                "sample_id": sample_id,
                "sample_index": idx,
                "reason": str(exc),
            })
            continue

        row = {
            "method_name": method_result["method_name"],
            "sample_id": sample_id,
            "sample_index": idx,
        }
        for parameter in PARAMETERS:
            row[f"{parameter}_true"] = true_params[parameter]
            row[f"{parameter}_pred"] = pred_params[parameter]
        row["true_decay_db"] = true_params["decay_db"]
        row["pred_decay_db"] = pred_params["decay_db"]
        rows.append(row)
    return pd.DataFrame(rows), pd.DataFrame(invalid)


def summarize_parameter_metrics(parameter_rows, invalid_rows, method_name):
    summary = {
        "method_name": method_name,
        "valid_parameter_samples": int(len(parameter_rows)),
        "invalid_parameter_samples": int(len(invalid_rows)),
    }
    for parameter in PARAMETERS:
        true_key = f"{parameter}_true"
        pred_key = f"{parameter}_pred"
        if len(parameter_rows) == 0:
            summary[f"{parameter}_mae"] = np.nan
            summary[f"{parameter}_rmse"] = np.nan
            summary[f"{parameter}_r2"] = np.nan
            continue
        true_values = parameter_rows[true_key].to_numpy(dtype=np.float64)
        pred_values = parameter_rows[pred_key].to_numpy(dtype=np.float64)
        summary[f"{parameter}_mae"] = mae(true_values, pred_values)
        summary[f"{parameter}_rmse"] = rmse(true_values, pred_values)
        summary[f"{parameter}_r2"] = r2_score(true_values, pred_values)
    return summary


In [ ]:
loaded_methods = load_all_results(RESULT_FILES, default_rate=SAMPLE_RATE) if RESULT_FILES else []

waveform_summaries = []
parameter_summaries = []
parameter_detail_frames = []
invalid_parameter_frames = []

for method_result in loaded_methods:
    waveform_summary = compute_waveform_metrics(method_result["y_true"], method_result["y_pred"])
    waveform_summary.update({
        "method_name": method_result["method_name"],
        "sample_rate": method_result["sample_rate"],
        "file_path": str(method_result["file_path"]),
    })
    waveform_summaries.append(waveform_summary)

    parameter_rows, invalid_rows = extract_parameter_rows(method_result)
    parameter_detail_frames.append(parameter_rows)
    invalid_parameter_frames.append(invalid_rows)
    if len(invalid_rows) > 0:
        warnings.warn(
            f"{method_result['method_name']}: skipped {len(invalid_rows)} sample(s) with invalid decay regions"
        )
    parameter_summaries.append(
        summarize_parameter_metrics(parameter_rows, invalid_rows, method_result["method_name"])
    )

waveform_summary_df = pd.DataFrame(waveform_summaries)
parameter_summary_df = pd.DataFrame(parameter_summaries)
parameter_details_df = pd.concat(parameter_detail_frames, ignore_index=True) if parameter_detail_frames else pd.DataFrame()
invalid_parameter_df = pd.concat(invalid_parameter_frames, ignore_index=True) if invalid_parameter_frames else pd.DataFrame()

if not waveform_summary_df.empty and not parameter_summary_df.empty:
    combined_summary_df = waveform_summary_df.merge(parameter_summary_df, on="method_name", how="inner")
elif not waveform_summary_df.empty:
    combined_summary_df = waveform_summary_df.copy()
else:
    combined_summary_df = pd.DataFrame()

print(f"Loaded methods: {len(loaded_methods)}")
if not loaded_methods:
    print("Add .npz files to RESULT_FILES and rerun the notebook.")


In [ ]:
display_cols_waveform = [
    "method_name",
    "waveform_mae",
    "waveform_rmse",
    "waveform_r2",
    "sample_mae_mean",
    "sample_mae_std",
    "sample_rmse_mean",
    "sample_rmse_std",
    "sample_r2_mean",
    "sample_r2_std",
    "n_samples",
]
display_cols_parameter = [
    "method_name",
    "valid_parameter_samples",
    "invalid_parameter_samples",
] + [
    f"{parameter}_{metric}"
    for parameter in PARAMETERS
    for metric in ("mae", "rmse", "r2")
]

if waveform_summary_df.empty:
    print("waveform_summary_df is empty.")
else:
    waveform_summary_df = waveform_summary_df.sort_values("waveform_rmse").reset_index(drop=True)
    display(waveform_summary_df[display_cols_waveform])

if parameter_summary_df.empty:
    print("parameter_summary_df is empty.")
else:
    parameter_summary_df = parameter_summary_df.sort_values("T60_rmse", na_position="last").reset_index(drop=True)
    display(parameter_summary_df[display_cols_parameter])

if combined_summary_df.empty:
    print("combined_summary_df is empty.")
else:
    combined_summary_df = combined_summary_df.sort_values("waveform_rmse").reset_index(drop=True)
    display(combined_summary_df)

if not invalid_parameter_df.empty:
    display(invalid_parameter_df)


In [ ]:
if PLOT_EXAMPLES and loaded_methods:
    example_method = loaded_methods[0]
    example_index = min(EXAMPLE_INDEX, example_method["y_true"].shape[0] - 1)
    sample_id = example_method["sample_id"][example_index]
    y_true_example = example_method["y_true"][example_index]
    y_pred_example = example_method["y_pred"][example_index]
    true_params = extract_acoustic_parameters(y_true_example, example_method["sample_rate"])
    pred_params = extract_acoustic_parameters(y_pred_example, example_method["sample_rate"])
    time_axis = np.arange(len(y_true_example)) / example_method["sample_rate"]

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
    axes[0].plot(time_axis, y_true_example, label="Ground Truth", linewidth=1.0)
    axes[0].plot(time_axis, y_pred_example, label="Prediction", linewidth=1.0, alpha=0.8)
    axes[0].set_title(f"RIR Example: {example_method['method_name']} | sample_id={sample_id}")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].legend()

    axes[1].plot(time_axis, true_params["decay_db"], label="Ground Truth EDC", linewidth=1.2)
    axes[1].plot(time_axis, pred_params["decay_db"], label="Prediction EDC", linewidth=1.2, alpha=0.8)
    axes[1].set_title("Schroeder Energy Decay Curves")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Decay (dB)")
    axes[1].legend()
    plt.show()
else:
    print("Plotting skipped because no methods were loaded or PLOT_EXAMPLES=False.")


In [ ]:
if EXPORT_CSV and not combined_summary_df.empty:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    waveform_path = OUTPUT_DIR / "measure_summary.csv"
    parameter_path = OUTPUT_DIR / "parameter_summary.csv"
    combined_path = OUTPUT_DIR / "combined_summary.csv"
    waveform_summary_df.to_csv(waveform_path, index=False)
    parameter_summary_df.to_csv(parameter_path, index=False)
    combined_summary_df.to_csv(combined_path, index=False)
    print(f"Saved waveform summary to {waveform_path.resolve()}")
    print(f"Saved parameter summary to {parameter_path.resolve()}")
    print(f"Saved combined summary to {combined_path.resolve()}")
else:
    print("CSV export skipped. Set EXPORT_CSV=True after loading result files.")
